In [0]:
dbutils.widgets.text(name="source_name", defaultValue="")
dbutils.widgets.text(name="file_format",defaultValue="")
dbutils.widgets.text(name="catalog",defaultValue="")
src_name=dbutils.widgets.get("source_name")
file_format=dbutils.widgets.get("file_format")
catalog_name=dbutils.widgets.get("catalog")
target_table=f"{catalog_name}.bronze.{source_name}_bronze"

In [0]:
print(file_format)

In [0]:
from pyspark.sql.functions import *

# Configuration
source_path        = f"/Volumes/olist_dev/bronze/raw_vol/{src_name}/"
checkpoint_path    = f"/Volumes/olist_dev/bronze/checkpoints/{src_name}/"
schema_location_path = f"/Volumes/olist_dev/bronze/checkpoints/{src_name}/schema/" # the reason here i use the olist_Dev for the data is that i am using the dev for the raw data location i know i should have make the seprate vilume outside the dev but this is a larning via major project practices

# 1. Read Stream with Autoloader
if file_format == "csv":
    df = (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", file_format)
        .option("cloudFiles.schemaLocation", schema_location_path)
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("header", "true")
        .load(source_path))

elif file_format == "json":
    df = (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", file_format)
        .option("cloudFiles.schemaLocation", schema_location_path)
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("multiLine", "true")
        .load(source_path))

else:  # parquet
    df = (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", file_format)
        .option("cloudFiles.schemaLocation", schema_location_path)
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .load(source_path))

# Add metadata
df = df.withColumn("ingestion_time", current_timestamp())

# 2. Write Stream to Delta Table
query = (df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(target_table))

# 3. Monitor the stream
query.awaitTermination()